# ggblab Examples

This notebook demonstrates basic usage of `ggblab`. It walks through loading a `.ggb` file, displaying it in the GeoGebra widget, and parsing the construction for inspection and manipulation.

Key points:

- Large constructions are handled using `getBase64` / `setBase64` (zip + base64).
- Run cells from top to bottom for the examples to work as intended.
- Cells that produce output (e.g., version checks, object lists) will display their results directly below the code cell.


In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

## Create and initialize GeoGebra

The next two cells import the `GeoGebra` controller and initialize the widget. The import cell does not print output; the initialization cell may open the widget and produce transient console output in the notebook UI.

## Create GeoGebra controller

Import and construct the `GeoGebra` controller; the next cell initializes the widget in the JupyterLab UI.

In [2]:
from ggblab import GeoGebra

## Initialize the GeoGebra widget

This cell runs `await GeoGebra().init()` to open the GeoGebra widget attached to the current kernel. Expect the widget to appear in the JupyterLab UI; the cell may print status messages or the kernel may emit frontend messages (these appear in the notebook output area).

In [3]:
# open GeoGebra Widget on left-side
ggb = await GeoGebra().init()

Using local cached file: xsd/common.xsd


## Version checks (package)

This cell prints the installed `ggblab` package version. Expect a short version string as the cell output.

In [4]:
# ggblab version
import ggblab
ggblab.__version__

'1.1.0'

## Version checks (applet)

This cell queries the GeoGebra applet with `getVersion()` and prints the applet version string. The output helps verify compatibility.

In [5]:
# GeoGebra Applet version
r = await ggb.function("getVersion")
r

'5.2.909.9'

## Load a .ggb file

Load a `.ggb` file from disk into the local `ggb.construction` object for inspection and sending to the applet.

In [6]:
# load from .ggb (zipped, and base64 encoded)
c = ggb.file.load('2025_06_08.ggb')

## Send construction to the applet

Use `setBase64` to send the loaded .ggb (zip+base64) to the GeoGebra view so it is rendered in the widget.

In [7]:
# sending loaded construction to GeoGebra view and draw
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

## Parse the construction XML

Decode the construction XML into Python structures (dicts/lists) so you can inspect and modify element attributes programmatically.

In [ ]:
# parse loaded xml as python dict
o = c.ggb_schema.decode(c.geogebra_xml)

In [ ]:
# list of all object names
[e['@label'] for e in o['element']]

In [ ]:
[e['@label'] for e in o['element']].index('A')

In [ ]:
o['element'][1]

In [ ]:
# change visible state of object 'A'
o['element'][1]['show'][0]['@object'] = False

In [ ]:
import xmlschema

In [ ]:
# construct attributes from dict
x = xmlschema.etree_tostring(c.ggb_schema.encode(o['element'][1], 'element'))
print(x)

## Temporary updates with `preserve()`

The `preserve()` context manager snapshots the applet state (using a zip+base64 snapshot) and restores it automatically when the block exits.

Usage notes:

- The snapshot is taken with `getBase64` and restored with `setBase64` when available.
- The yielded `snap` object includes `base64_zip`, `xml` (optional), `timestamp`, `size_bytes`, and `sha1`.
- Call `await snap.restore()` to restore immediately; call `snap.release()` to drop the stored snapshot from memory.

In [ ]:
import asyncio

In [ ]:
# update the applet
async with ggb.preserve() as snap:
    r = await ggb.function("evalXML", [x])
    await asyncio.sleep(3)

## Advanced inspection

The following examples show how to decode the base64 snapshot, open the embedded zip, and inspect internal files (including the main construction XML). Expected outputs include a list of filenames inside the `.ggb` zip and the decoded XML string(s).

## Inspect base64 snapshot

Decode the base64 snapshot and inspect the zipped content (files inside the .ggb). This is useful when debugging or extracting the main construction XML.

In [ ]:
import base64
import zipfile
import io

In [ ]:
type(snap.base64_zip)

In [ ]:
with zipfile.ZipFile(io.BytesIO(base64.b64decode(snap.base64_zip))) as zf:
    for info in zf.infolist():
        if info.filename == 'geogebra.xml':
            with zf.open(info) as f:
                xml = f.read()

In [ ]:
type(xml.decode())

In [ ]:
# r = await ggb.function("setXML", [xml.decode()])

In [ ]:
await ggb.function("setBase64", [snap.base64_zip])

## draw step-by-step from new construction

In [ ]:
# draw step-by-step from new construction
r = await ggb.function("newConstruction")

names = [e['@label'] for e in o['element']]
for n in names:
    cmd = None
    ci, co = None, None
    for _c in o['command']:
        _ci = tuple(zip(*_c['input'].items()))[1]
        _co = tuple(zip(*_c['output'].items()))[1]
        if n in _co:
            cmd = _c
            ci = _ci
            co = _co
            break

    match names.index(n):
        case int(i):
            ei = o['element'][i]
            if cmd:
                if co.index(n) == 0:
                    x = xmlschema.etree_tostring(c.ggb_schema.encode(cmd, 'construction/command'))
                    r = await ggb.function("evalXML", [x])
                    # print(f"command: {n}")
            x = xmlschema.etree_tostring(c.ggb_schema.encode(ei, 'element'))
            r = await ggb.function("evalXML", [x])
            # print(f"element:{n}")

In [ ]:
ggb.comm.logs

In [ ]:
ggb.comm.recv_msgs

In [ ]:
ggb.comm.pending_futures

In [ ]:
ggb.comm.recv_events.queue

## ggblab_extra — construction protocol from the applet

These cells demonstrate how to extract a construction protocol (objects, commands, and dependency metadata) from a running GeoGebra applet using `ggblab_extra`.

They show how to initialize a structured `DataFrame` from the applet (`ConstructionIO.initialize_dataframe(ggb, use_applet=True)`) so you can analyze element attributes, sequences, and dependencies programmatically. These are analysis helpers and do not alter the stored applet state unless you explicitly call functions that modify the applet.

In [8]:
import polars as pl
import networkx as nx

In [69]:
from ggblab_extra.construction_io import ConstructionIO
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [70]:
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
0,"""O_{2}""","""point""",null,"""O_{2} = (-2.1, -0.1)""",null,9,true,true,false
1,"""A""","""point""",null,"""A = (10, 0)""",null,2,true,true,false
2,"""B""","""point""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",null,9,true,false,false
3,"""c""","""circle""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",null,9,true,false,false
4,"""n""","""line""","""Line(O_{2}, A)""","""n: y = -0.1""",null,4,false,false,false
5,"""O'""","""point""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",null,4,false,true,false
6,"""p""","""circle""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",null,4,false,false,false
7,"""j""","""line""","""Tangent(A, c)""","""j: -5.8x - 7.6y = -58.2""","""$a+c$""",2,true,false,false
8,"""l""","""line""","""Tangent(A, c)""","""l: 5.9x - 7.5y = 59.4""",null,2,true,false,false


## ggblab_extra — construction tree parser

Use `ConstructionTreeParser` to build a graph of construction dependencies and perform graph analyses. The examples below show how to:

- Create the parser with `p = ConstructionTreeParser(df)`
- Build the full dependency graph with `g1 = p.parse()`
- Extract bounded subgraphs with `g2 = p.parse_subgraph(max_depth=3)`

These operations are for inspection and analysis; they help you visualize and trace construction dependency paths.

In [71]:
from ggblab_extra.construction_parser import ConstructionTreeParser

In [72]:
p = ConstructionTreeParser(df)

In [73]:
g1 = p.parse()

In [74]:
labels_map = {}
for n, t in p.df["Name", "Type"].rows():
    g1.nodes[n]["label"] = f"{n} ({t})"

In [75]:
nx.write_network_text(g1)

╟── O_{2} (point)
╎   ├─╼ B (point) ╾ A (point)
╎   │   └─╼ c (circle) ╾ O_{2} (point)
╎   │       ├─╼ j (line) ╾ A (point)
╎   │       │   ├─╼ q (line) ╾ l (line)
╎   │       │   ├─╼ r (line) ╾ l (line)
╎   │       │   │   └─╼ O_{1} (point)
╎   │       │   │       ├─╼ t (line) ╾ l (line)
╎   │       │   │       │   ├─╼ F_{1} (point) ╾ l (line)
╎   │       │   │       │   │   ├─╼ h_1 (circle) ╾ O_{1} (point)
╎   │       │   │       │   │   │   ├─╼ p_2 (circle) ╾ O_{2} (point), c (circle)
╎   │       │   │       │   │   │   │   ├─╼ q_2 (line) ╾ O_{1} (point)
╎   │       │   │       │   │   │   │   │   └─╼ N (point) ╾ p_2 (circle)
╎   │       │   │       │   │   │   │   │       └─╼ d_2 (ray) ╾ O_{2} (point)
╎   │       │   │       │   │   │   │   │           └─╼ T (point) ╾ c (circle)
╎   │       │   │       │   │   │   │   ├─╼ r_2 (line) ╾ O_{1} (point)
╎   │       │   │       │   │   │   │   │   └─╼ M (point) ╾ p_2 (circle)
╎   │       │   │       │   │   │   │   │       └─╼ b_2 (ray) 

In [16]:
g2 = p.parse_subgraph(max_depth=3)

parse_subgraph: timeout exceeded during candidate evaluation (10.00s), falling back to parse_subgraph_legacy


In [17]:
nx.write_network_text(g2)

╟── O_{2}
╎   ├─╼ B ╾ A
╎   │   └─╼ c
╎   │       ├─╼ l
╎   │       │   └─╼ r ╾ j
╎   │       │       └─╼ O_{1}
╎   │       │           ├─╼ t
╎   │       │           │   └─╼ F_{1}
╎   │       │           │       └─╼ h_1
╎   │       │           │           ├─╼ k_1
╎   │       │           │           ├─╼ p_2
╎   │       │           │           │   ├─╼ r_2
╎   │       │           │           │   └─╼ q_2
╎   │       │           │           │       └─╼ N
╎   │       │           │           ├─╼ j_1
╎   │       │           │           └─╼ k_2
╎   │       │           │               ├─╼ m_2
╎   │       │           │               └─╼ n_2
╎   │       │           ├─╼ O_3
╎   │       │           │   └─╼ p_1
╎   │       │           │       ├─╼ D
╎   │       │           │       ├─╼ E
╎   │       │           │       └─╼ C
╎   │       │           └─╼ b
╎   │       │               ├─╼ P'
╎   │       │               └─╼ P
╎   │       ├─╼ j
╎   │       │   └─╼  ...
╎   │       ├─╼ G ╾ p
╎   │       └─╼ 

In [18]:
p.df.filter(~pl.col("Auxiliary"))["Sequence", "Type", "Name", "Command", "Value", "Layer", "DependsOn"]

Sequence,Type,Name,Command,Value,Layer,DependsOn
u32,str,str,str,str,u32,list[str]
0,"""point""","""O_{2}""",null,"""O_{2} = (-2.1, -0.1)""",9,[]
1,"""point""","""A""",null,"""A = (10, 0)""",2,[]
2,"""point""","""B""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",9,"[""A"", ""O_{2}""]"
3,"""circle""","""c""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",9,"[""A"", ""B"", ""O_{2}""]"
4,"""line""","""n""","""Line(O_{2}, A)""","""n: y = -0.1""",4,"[""A"", ""O_{2}""]"
5,"""point""","""O'""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",4,"[""A"", ""O_{2}""]"
6,"""circle""","""p""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",4,"[""A"", ""O'"", ""O_{2}""]"
7,"""point""","""G""","""Intersect(c, p)""","""G = (2.4, 5.8)""",2,"[""A"", ""B"", … ""p""]"
8,"""point""","""F_{2}""","""Intersect(c, p)""","""F_{2} = (2.5, -5.9)""",2,"[""A"", ""B"", … ""p""]"


In [19]:
# find root
[node for node, degree in p.G.in_degree() if degree == 0]

['O_{2}', 'A']

In [20]:
p.roots, p.leaves

(['O_{2}', 'A'],
 ['n',
  'β',
  'γ',
  'a',
  'c_1',
  'g_1',
  'a_1',
  'c_2',
  'i_1',
  'k',
  'm',
  'δ',
  'q',
  'θ',
  'q_1',
  'r_1',
  'd',
  'o',
  'l_2',
  'α',
  'f',
  'i',
  'c_3',
  'i_2',
  'a_2',
  'h',
  'g',
  'f_1',
  's_1',
  'd_1',
  'e_1',
  'F',
  'f_2',
  'e_2',
  'o_3',
  'o_{2}',
  's_2',
  't_2',
  'b_2',
  'T',
  'ι',
  'κ',
  'λ',
  'μ',
  'U',
  'h_3',
  'i_3',
  'ν',
  'η',
  'ζ',
  'ε'])

## Utilities and analysis

These cells provide small utilities for working with constructions and snapshots produced by `ggblab` and `ggblab_extra`.

What these do:

- Save or export snapshots (base64 zip) to disk for later inspection.
- Decode base64 snapshots and inspect the embedded `.ggb` zip contents (including `geogebra.xml`).
- Provide graph-analysis helpers (e.g., longest-path, shortest-path length) to inspect construction dependency graphs built by `ConstructionTreeParser`.
- Offer lightweight helpers for exploring `ggb` applet internals (comm logs, pending messages, etc.).

Important: these cells focus on analysis and helper utilities — they do not perform automatic cleanup or remove application state.

In [21]:
def get_longest_path_to_node(G, target_node):
    roots = [node for node, degree in G.in_degree() if degree == 0]
    longest_path = []
    max_len = 0
    for root in roots:
        for path in nx.all_simple_paths(G, root, target_node):
            if len(path) > max_len:
                max_len = len(path)
                longest_path = path
    return longest_path

In [22]:
path = get_longest_path_to_node(p.G, 'I')
path

['O_{2}', 'B', 'c', 'j', 'r', 'O_{1}', 't', 'F_{1}', 'h_1', 'k_1', 'I']

In [23]:
depths = nx.shortest_path_length(p.G, source='A')
depths

{'A': 0,
 'B': 1,
 'n': 1,
 "O'": 1,
 'p': 1,
 'j': 1,
 'l': 1,
 'β': 1,
 'γ': 1,
 't1': 1,
 'a': 1,
 'g_1': 1,
 't2': 1,
 'a_1': 1,
 'i_1': 1,
 'k': 1,
 'm': 1,
 'δ': 1,
 'θ': 1,
 'q_1': 1,
 'r_1': 1,
 't4': 1,
 'c_3': 1,
 'i_2': 1,
 'g': 1,
 'f_1': 1,
 's_1': 1,
 'η': 1,
 'ζ': 1,
 'ε': 1,
 'c': 2,
 'G': 2,
 'F_{2}': 2,
 'q': 2,
 'r': 2,
 "P'": 2,
 'P': 2,
 't': 2,
 'F_{1}': 2,
 'C': 2,
 'f': 2,
 'e_1': 2,
 'c_1': 2,
 'c_2': 2,
 'a_2': 2,
 'I': 3,
 'k_2': 3,
 'p_2': 3,
 'T': 3,
 'O': 3,
 'i': 3,
 'h': 3,
 'd_1': 3,
 'ι': 3,
 'μ': 3,
 'O_{1}': 3,
 'j_1': 3,
 'k_1': 3,
 'l_1': 3,
 'm_1': 3,
 'E': 3,
 'F': 3,
 'h_1': 3,
 'κ': 3,
 'λ': 3,
 'ν': 3,
 'm_2': 4,
 'n_2': 4,
 'L': 4,
 'S': 4,
 'q_2': 4,
 'r_2': 4,
 'M': 4,
 'N': 4,
 'b': 4,
 'O_3': 4,
 't3': 4,
 'd': 4,
 'o': 4,
 'α': 4,
 'Q': 4,
 'R': 4,
 'h_3': 4,
 'i_3': 4,
 'U': 4,
 'f_2': 4,
 't5': 4,
 'o_3': 4,
 'o_{2}': 4,
 's_2': 5,
 't_2': 5,
 'b_2': 5,
 'd_2': 5,
 'p_1': 5,
 'e_2': 5,
 'l_2': 5,
 'D': 6}

In [24]:
depths = nx.shortest_path_length(p.G2, source='A')
depths

{'A': 0,
 'B': 1,
 "O'": 1,
 'c': 2,
 'p': 2,
 'l': 3,
 'j': 3,
 'G': 3,
 'F_{2}': 3,
 'r': 4,
 'O_{1}': 5,
 't': 6,
 'O_3': 6,
 'b': 6,
 'F_{1}': 7,
 'p_1': 7,
 "P'": 7,
 'P': 7,
 'h_1': 8,
 'D': 8,
 'E': 8,
 'C': 8,
 'k_1': 9,
 'p_2': 9,
 'j_1': 9,
 'k_2': 9,
 'r_2': 10,
 'q_2': 10,
 'm_2': 10,
 'n_2': 10,
 'N': 11}

In [25]:
prev = set(list(zip(*(list(p.G.in_edges('R')) + list(p.G.in_edges('S')))))[0])
prev

{'h_1', 'k_1', 'k_2', 'n_2'}

In [26]:
[(n, depths[n]) for n in depths if n in prev]

[('h_1', 8), ('k_1', 9), ('k_2', 9), ('n_2', 10)]

In [27]:
nx.write_network_text(p.roots_to_targets_with_containers('R'))

╟── O_{2} (point)
╎   ├─╼ B (point) ╾ A (point)
╎   │   └─╼ c (circle) ╾ O_{2} (point)
╎   │       ├─╼ j (line) ╾ A (point)
╎   │       │   ├─╼ r (line) ╾ l (line)
╎   │       │   │   └─╼ O_{1} (point)
╎   │       │   │       ├─╼ b (line) ╾ O_{2} (point)
╎   │       │   │       │   └─╼ P' (point) ╾ j (line)
╎   │       │   │       │       └─╼ k_1 (line) ╾ h_1 (circle)
╎   │       │   │       │           └─╼ R (point) ╾ h_1 (circle)
╎   │       │   │       ├─╼ t (line) ╾ l (line)
╎   │       │   │       │   └─╼ F_{1} (point) ╾ l (line)
╎   │       │   │       │       └─╼ h_1 (circle) ╾ O_{1} (point)
╎   │       │   │       │           └─╼  ...
╎   │       │   │       └─╼  ...
╎   │       │   └─╼  ...
╎   │       └─╼ l (line) ╾ A (point)
╎   │           └─╼  ...
╎   └─╼  ...
╙── A (point)
    └─╼  ...


## .

In [28]:
p.df.filter(pl.col("Layer").is_in([9]))["Sequence", "Type", "Name", "Command", "Value", "Layer", "DependsOn"]

Sequence,Type,Name,Command,Value,Layer,DependsOn
u32,str,str,str,str,u32,list[str]
0,"""point""","""O_{2}""",null,"""O_{2} = (-2.1, -0.1)""",9,[]
2,"""point""","""B""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",9,"[""A"", ""O_{2}""]"
3,"""circle""","""c""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",9,"[""A"", ""B"", ""O_{2}""]"
26,"""point""","""O_{1}""","""Point(r)""","""O_{1} = (10, -4.7)""",9,"[""A"", ""B"", … ""r""]"
33,"""circle""","""h_1""","""Circle(O_{1}, F_{1})""","""h_1: (x - 10)² + (y + 4.7)² = …",9,"[""A"", ""B"", … ""t""]"


## Geometric scenes

In [91]:
l1 = p.df.filter(pl.col("Layer").is_in([9, 0, 1, 2, 3]))["Name"].to_list()
l1

['O_{2}',
 'A',
 'B',
 'c',
 'j',
 'l',
 'q',
 'r',
 'O_{1}',
 't',
 'F_{1}',
 'h_1',
 'p_2',
 'q_2',
 'r_2',
 'M',
 'b_2',
 'F_{2}',
 "P'",
 'P',
 'j_1',
 'k_1',
 'l_1',
 'm_1',
 'C',
 'q_1',
 'Q',
 'R',
 'h',
 'e_1',
 'I',
 'k_2',
 'm_2',
 'n_2',
 'L',
 'N',
 'S',
 's_2',
 't_2',
 'd_2',
 'T',
 'ι',
 'κ',
 'λ',
 'μ',
 'U',
 'h_3',
 'i_3',
 'η',
 'ζ',
 'ε']

In [92]:
nx.write_network_text(p.G.subgraph(l1))

╟── O_{2} (point)
╎   ├─╼ B (point) ╾ A (point)
╎   │   └─╼ c (circle) ╾ O_{2} (point)
╎   │       ├─╼ j (line) ╾ A (point)
╎   │       │   ├─╼ q (line) ╾ l (line)
╎   │       │   ├─╼ r (line) ╾ l (line)
╎   │       │   │   └─╼ O_{1} (point)
╎   │       │   │       ├─╼ t (line) ╾ l (line)
╎   │       │   │       │   └─╼ F_{1} (point) ╾ l (line)
╎   │       │   │       │       ├─╼ h_1 (circle) ╾ O_{1} (point)
╎   │       │   │       │       │   ├─╼ p_2 (circle) ╾ O_{2} (point), c (circle)
╎   │       │   │       │       │   │   ├─╼ q_2 (line) ╾ O_{1} (point)
╎   │       │   │       │       │   │   │   └─╼ N (point) ╾ p_2 (circle)
╎   │       │   │       │       │   │   │       └─╼ d_2 (ray) ╾ O_{2} (point)
╎   │       │   │       │       │   │   │           └─╼ T (point) ╾ c (circle)
╎   │       │   │       │       │   │   ├─╼ r_2 (line) ╾ O_{1} (point)
╎   │       │   │       │       │   │   │   └─╼ M (point) ╾ p_2 (circle)
╎   │       │   │       │       │   │   │       └─╼ b_2 (ray) 

In [93]:
nx.ancestors(p.G, 'F_{2}')

{'A',
 'B',
 'F_{1}',
 'M',
 'O_{1}',
 'O_{2}',
 'b_2',
 'c',
 'h_1',
 'j',
 'l',
 'p_2',
 'r',
 'r_2',
 't'}

In [94]:
p.ft['F_{2}']

['Intersect', 'c', 'b_2', '1']

In [95]:
p.ft['p']

['Circle', "O'", 'A']

In [96]:
from itertools import zip_longest

In [97]:
l0 = range(10)
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=False)))
l0 = [9, 0, 1, 2, 3]
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=True)))

[None, None, None, None, None]

In [98]:
p.roots

['O_{2}', 'A']

In [99]:
p.G.subgraph(l1)

In [100]:
from collections import defaultdict
s0 = defaultdict(int)
for n, e in nx.dfs_edges(p.G.subgraph(l1), source='A'):
    # print(nx.ancestors(p.G, n), e)
    for n1 in nx.ancestors(p.G, n):
        if n1 not in s0:
            print(n1)
            s0[n1] += 1
    s0[n] += 1
    s0[e] += 1
s0

O_{2}
l
b
P'
P


defaultdict(int,
            {'A': 1,
             'B': 2,
             'O_{2}': 1,
             'c': 3,
             'j': 4,
             'q': 1,
             'r': 2,
             'l': 4,
             'O_{1}': 2,
             't': 2,
             'F_{1}': 5,
             'h_1': 7,
             'p_2': 3,
             'q_2': 2,
             'N': 2,
             'd_2': 2,
             'T': 1,
             'r_2': 2,
             'M': 2,
             'b_2': 2,
             'F_{2}': 5,
             'h': 1,
             'e_1': 1,
             'ι': 1,
             'μ': 1,
             'j_1': 2,
             'b': 1,
             "P'": 2,
             'Q': 2,
             'η': 1,
             'k_1': 5,
             'R': 2,
             'λ': 1,
             'I': 2,
             'κ': 1,
             'h_3': 1,
             'i_3': 1,
             'l_1': 2,
             'P': 2,
             'U': 1,
             'm_1': 1,
             'k_2': 3,
             'm_2': 2,
             'L': 2,
            

In [101]:
from itertools import zip_longest

In [102]:
import asyncio

l0 = range(10)
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=False)))

l1 = list(s0.keys())

In [103]:
# for i in range(len(l1)):
#     l2 = list(zip_longest(l1, [True]*i, fillvalue=False))
#     # print(l2)
#     await ggb.function("setVisible", l2)
#     await asyncio.sleep(1)

In [104]:
%gui asyncio

In [105]:
# Helper function to create a future that waits for a widget change
# def wait_for_change(widget, value):
#     future = asyncio.Future()
#     def get_value(change):
#         future.set_result(change.new)
#         widget.unobserve(get_value, value)
#     widget.observe(get_value, value)
#     return future

# value = await wait_for_change(slider, 'value')

In [106]:
import ipywidgets as widgets
from IPython.display import display
import asyncio

# Your list of items
data_list = list(s0.keys())

# Create an IntSlider
# The min value is 0 (the first index)
# The max value is the last index (length of list - 1)
# The default value can be set to the starting index
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(data_list) - 1,
    step=1,
    description='Select Index:',
    continuous_update=True # Update only when the slider is released
)

# Function to handle changes in the slider's value
async def on_slider_change(change):
    selected_index = change['new']
    selected_item = data_list[selected_index]
    l2 = list(zip_longest(l1, [True]*(selected_index+1), fillvalue=False))
    await(ggb.function("setVisible", l2))

def observe_wrapper(change):
    asyncio.ensure_future(on_slider_change(change))

slider.observe(observe_wrapper, names='value')

display(slider)
# Initial display (optional, can be called once to show the initial state)
await on_slider_change({'new': slider.value})

IntSlider(value=0, description='Select Index:', max=51)

## store

In [88]:
ggb.file.base64_buffer = await ggb.function("getBase64")
c.save(overwrite=True)